# 02b - Preprocessing of the full ESM25 dataset

Applies to every ESM25 record the preprocessing chosen on the development subset in notebook 02 (window, anchor,
resampling, spectrogram and scalogram on the common grid; parameters in the `preprocessing` section of
`configs/default.yaml`). No exploratory analysis is repeated here.

- Metadata and traces are read from the full SeisBench datasets in the cache, whatever `data.source` and
  `waveforms.source` say (so notebooks 01 and 02 can stay on the development subset).
- Window placement (`preprocessing.window.placement`): `anchored` (anchor always at `pre_anchor_s` from the window
  start, zeros where the window leaves the recording) or `clamped` (the same window shifted as little as possible to
  lie inside the recorded data). Each placement has its own output, so the two can coexist:
  `data/processed/seisbench/features.h5` (anchored) and `data/processed/seisbench/clamped/features.h5`, each with its
  `records.csv`. To train on it, set `training.feature_source: seisbench` and the same placement.
- The run is **resumable**: if it stops (kernel restart, sleep, error), run the notebook again and it continues from
  the records not yet processed. If the file is already complete, the preprocessing is skipped.
- At the end: checks on the stored records and comparison with the development subset preprocessed with the same
  placement, if it exists (the same records must give the same features).

## 0. Setup

In [2]:
from dataclasses import replace
from pathlib import Path
import shutil

import numpy as np
import pandas as pd

from wqu.config import load_config
from wqu.data.exploration import numeric_summary_by_label
from wqu.data.loading import CV_COMPONENTS, load_esm25_metadata
from wqu.data.waveforms import iter_waveform_batches, stored_sampling_rate_hz
from wqu.preprocessing.pipeline import (
    FEATURE_INDEX_COLUMN,
    REPRESENTATIONS,
    feature_file_path,
    grid_from_config,
    preprocess_and_store,
    read_feature_records,
    read_features,
)

CONFIG_PATH = Path("..") / "configs" / "default.yaml"
FULL_SOURCE = "seisbench"
DEV_SOURCE = "dev_subset"
BYTES_PER_VALUE = 2
COMPARISON_BLOCK = 250

# Metadata and traces from the SeisBench datasets, whatever data.source and waveforms.source say.
config = load_config(CONFIG_PATH)
config = replace(
    config,
    data=replace(config.data, source=FULL_SOURCE),
    waveforms=replace(config.waveforms, source=FULL_SOURCE),
)
columns = config.columns
preprocessing = config.preprocessing
label_names = {subset.label: subset.name for subset in config.data.subsets}
grid = grid_from_config(preprocessing)

_, metadata = load_esm25_metadata(config)
sampling_rate_hz = stored_sampling_rate_hz(metadata)
features_path = feature_file_path(preprocessing, FULL_SOURCE)
print(f"Records: {len(metadata):,}; window {preprocessing.pre_anchor_s:g} s before to {preprocessing.post_anchor_s:g} s "
      f"after the anchor ({preprocessing.anchor_method}, {preprocessing.window_placement} placement); "
      f"grid {len(grid.frequencies_hz)} x {len(grid.times_s)}")
print(f"Output: {features_path}")

/Users/giulianaparadiso/miniconda3/envs/wqu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Records: 100,249; window 30 s before to 90 s after the anchor (envelope, clamped placement); grid 128 x 128
Output: /Users/giulianaparadiso/Desktop/INGV Projects/waveform-quality-uncertainty/data/processed/seisbench/clamped/features.h5


## 1. Disk space

Size of the feature file (both representations, float16) against the free space on the disk.

In [3]:
values_per_record = len(REPRESENTATIONS) * len(CV_COMPONENTS) * len(grid.frequencies_hz) * len(grid.times_s)
required_bytes = len(metadata) * values_per_record * BYTES_PER_VALUE
features_path.parent.mkdir(parents=True, exist_ok=True)
free_bytes = shutil.disk_usage(features_path.parent).free
print(f"Feature file: {required_bytes / 1e9:.1f} GB; free space: {free_bytes / 1e9:.1f} GB")
if not features_path.exists() and free_bytes < required_bytes:
    raise RuntimeError("Not enough free disk space for the feature file.")

Feature file: 19.7 GB; free space: 55.8 GB


## 2. Preprocessing (resumable)

About two hours on a laptop. The progress bar starts from the records already processed in a previous run.

In [4]:
if features_path.exists():
    stored_records = read_feature_records(features_path, columns.text)
    print(f"Feature file already complete: {len(stored_records):,} records")
else:
    stored_records = preprocess_and_store(
        metadata, lambda selected: iter_waveform_batches(selected, config), sampling_rate_hz, preprocessing,
        config.waveforms.zero_tolerance, features_path,
    )
    print(f"Stored {len(stored_records):,} records in {features_path}")

Feature file already complete: 100,249 records


## 3. Checks on the stored records

Records per class, window statistics by class (to compare with notebook 02 on the development subset) and share of
records with particular windows.

In [5]:
print(stored_records[columns.label].map(label_names).value_counts().to_string())
WINDOW_COLUMNS = ["anchor_time_s", "recorded_fraction_kept", "energy_fraction_kept",
                  *[f"missing_in_window_s_{component}" for component in CV_COMPONENTS]]
numeric_summary_by_label(stored_records, WINDOW_COLUMNS, columns.label).rename(index=label_names, level=columns.label).round(3)

label
good    64213
bad     36036


count     mean     std    min     25%     50%  \
column                 label                                                    
anchor_time_s          good   64213.0   76.058  39.581  1.000  61.995  81.090   
                       bad    36036.0  105.414  72.616  0.000  71.996  89.960   
recorded_fraction_kept good   64213.0    0.547   0.248  0.286   0.403   0.434   
                       bad    36036.0    0.477   0.205  0.000   0.387   0.418   
energy_fraction_kept   good   64213.0    0.925   0.138  0.000   0.921   0.990   
                       bad    36036.0    0.674   0.233  0.000   0.468   0.659   
missing_in_window_s_u  good   64213.0    9.090  22.530  0.000   0.000   0.000   
                       bad    36036.0    4.630  18.288  0.000   0.000   0.000   
missing_in_window_s_v  good   64213.0    9.093  22.532  0.000   0.000   0.000   
                       bad    36036.0    4.775  18.788  0.000   0.000   0.000   
missing_in_window_s_w  good   64213.0    9.093  22.532  0.000   0.000   0.000   
                       bad    36036.0    4.852  18.999  0.000   0.000   0.000   

                                  75%      max  
column                 label                    
anchor_time_s          good    93.055  413.580  
                       bad    117.470  413.165  
recorded_fraction_kept good     0.667    1.000  
                       bad      0.441    1.000  
energy_fraction_kept   good     1.000    1.000  
                       bad      0.904    1.000  
missing_in_window_s_u  good     0.000  115.065  
                       bad      0.000  120.000  
missing_in_window_s_v  good     0.000  115.040  
                       bad      0.000  120.000  
missing_in_window_s_w  good     0.000  115.030  
                       bad      0.000  120.000

In [6]:
window_samples = stored_records["window_stop"] - stored_records["window_start"]
missing = stored_records[[f"missing_in_window_s_{component}" for component in CV_COMPONENTS]]
window_flags = pd.DataFrame(
    {
        "window starts before the recording": stored_records["window_start"] < 0,
        "some data missing in the window": missing.max(axis=1) > 0,
        "a component without data in the window": missing.max(axis=1) >= preprocessing.window_length_s,
        "energy kept < 95%": stored_records["energy_fraction_kept"] < 0.95,
    }
)
if "window_shift_s" in stored_records:
    window_flags["window shifted from the anchored position"] = stored_records["window_shift_s"] != 0
window_flags.groupby(stored_records[columns.label]).mean().T.rename(columns=label_names).round(3)

label,good,bad
window starts before the recording,0.139,0.055
some data missing in the window,0.189,0.107
a component without data in the window,0.000,0.008
energy kept < 95%,0.309,0.806
window shifted from the anchored position,0.139,0.280


With clamped windows: shift of the shifted windows (positive = later than the anchored window) and position of
the anchor in the window (anchored windows: always `pre_anchor_s`).

In [7]:
if "window_shift_s" in stored_records:
    shifted = stored_records[stored_records["window_shift_s"] != 0]
    print(f"Shifted windows: {len(shifted):,} of {len(stored_records):,}")
    shift_summary = numeric_summary_by_label(shifted, ["window_shift_s", "anchor_offset_s"], columns.label)
    display(shift_summary.rename(index=label_names, level=columns.label).round(2))
else:
    print("The record table has no window_shift_s column (file written before the placement option).")

Shifted windows: 19,014 of 100,249


count   mean    std   min    25%    50%    75%    max
column          label                                                         
window_shift_s  good    8908.0  -7.56  24.08 -90.0 -14.67  -2.72   7.30   30.0
                bad    10106.0 -24.90  39.40 -90.0 -59.53 -19.08   8.32   30.0
anchor_offset_s good    8908.0  37.56  24.08   0.0  22.70  32.72  44.67  120.0
                bad    10106.0  54.90  39.40   0.0  21.68  49.08  89.53  120.0

In [8]:
missing_columns = [f"missing_in_window_s_{component}" for component in CV_COMPONENTS]
empty_components = (stored_records[missing_columns] >= preprocessing.window_length_s).sum(axis=1)
empty_counts = pd.crosstab(
    empty_components.rename("components without data in the window"),
    stored_records[columns.label].map(label_names),
    margins=True,
    margins_name="total",
)
empty_counts

label,bad,good,total
components without data in the window,,,
0,35737,64213,99950
1,93,0,93
2,23,0,23
3,183,0,183
total,36036,64213,100249


## 4. Comparison with the development subset

The records of the development subset are also in the full dataset: their windows and features must be the same in
the two files (the traces come from the same SeisBench datasets).

In [9]:
def compare_with_development(dev_path: Path) -> pd.Series:
    """Print the share of equal window starts and return the largest feature difference per representation."""
    dev_records = read_feature_records(dev_path, columns.text)
    full_rows = stored_records.loc[dev_records.index, FEATURE_INDEX_COLUMN].to_numpy()
    dev_rows = dev_records[FEATURE_INDEX_COLUMN].to_numpy()
    same_window = (stored_records.loc[dev_records.index, "window_start"].to_numpy()
                   == dev_records["window_start"].to_numpy())
    print(f"Development records: {len(dev_records):,}; same window start: {same_window.mean():.4f}")
    differences = {}
    for representation in REPRESENTATIONS:
        largest = 0.0
        for start in range(0, len(dev_rows), COMPARISON_BLOCK):
            stop = start + COMPARISON_BLOCK
            dev_values, _ = read_features(dev_path, representation, dev_rows[start:stop])
            full_values, _ = read_features(features_path, representation, full_rows[start:stop])
            largest = max(largest, float(np.abs(dev_values - full_values).max()))
        differences[representation] = largest
    return pd.Series(differences, name="largest difference (dB)")


dev_path = feature_file_path(preprocessing, DEV_SOURCE)
if dev_path.exists():
    display(compare_with_development(dev_path))
else:
    print(f"No development features with {preprocessing.window_placement} windows ({dev_path}): comparison skipped.")

Development records: 2,000; same window start: 1.0000


spectrogram    0.0
scalogram      0.0
Name: largest difference (dB), dtype: float64